In [1]:
import pandas as pd
import numpy as np
import os

import sys

# use absolute path here
project_path = "/mnt/School/PhD/AI221/Project/"
sys.path.insert(0, project_path)

from src.data_extraction.utils.constants import data_path

import warnings
warnings.filterwarnings('ignore')

In [2]:
fcp_folder = "flood_control_projects"
df = pd.read_csv(
    os.path.join(data_path, fcp_folder, "flood_control_per_city.csv")
)

In [4]:
df.CompletionDateActual.min()

'2022-07-01'

In [3]:
# Combine descriptions and make lowercase for robust keyword searching
df['full_desc'] = (df['ProjectDescription'].fillna('') + " " + 
                   df['ProjectComponentDescription'].fillna('')).str.lower()

# --- 1. Intervention Type ---
df['action_construction'] = df['full_desc'].str.contains('construct', na=False).astype(int)
df['action_rehabilitation'] = df['full_desc'].str.contains('rehab|restor|retrofit', na=False).astype(int)
df['action_repair'] = df['full_desc'].str.contains('repair', na=False).astype(int)
df['action_improvement'] = df['full_desc'].str.contains('improv|upgrad', na=False).astype(int)
df['action_extension'] = df['full_desc'].str.contains('exten|continu', na=False).astype(int)

# --- 2. Infrastructure Category ---
df['asset_drainage_system'] = df['full_desc'].str.contains('drainage|catch basin|box culvert', na=False).astype(int)
df['asset_slope_protection'] = df['full_desc'].str.contains('slope protection|rockfall|soil nailing', na=False).astype(int)
df['asset_bank_protection'] = df['full_desc'].str.contains('bank protection|revetment|gabion', na=False).astype(int)
df['asset_linear_canal'] = df['full_desc'].str.contains('line canal|lined canal|earth canal|open canal', na=False).astype(int)
df['asset_dike_levee'] = df['full_desc'].str.contains('dike|levee', na=False).astype(int)
df['asset_coastal_defense'] = df['full_desc'].str.contains('seawall|breakwater|coastal', na=False).astype(int)
df['asset_active_control'] = df['full_desc'].str.contains('pumping station|floodgate|sluice gate|pump', na=False).astype(int)

# --- 3. Protected Entity ---
df['protects_roadway'] = df['full_desc'].str.contains(r'\b(?:highway|road|avenue|street|st\.|blvd|boulevard|daang)\b', na=False).astype(int)
df['protects_education'] = df['full_desc'].str.contains('school|university|college|campus', na=False).astype(int)
df['protects_residential'] = df['full_desc'].str.contains('home|village|subdivision|housing|residen', na=False).astype(int)
df['protects_agricultural'] = df['full_desc'].str.contains(r'\bcis\b|farm|irrig', na=False).astype(int)

# --- 4. Water Body Typology ---
df['waterbody_river'] = df['full_desc'].str.contains('river', na=False).astype(int)
df['waterbody_creek'] = df['full_desc'].str.contains('creek|estero|stream', na=False).astype(int)
df['waterbody_coastal'] = df['full_desc'].str.contains(r'\b(sea|bay|gulf|coast|shore)\b', na=False).astype(int)

# Derived: If no natural water body is mentioned but drainage/roads are, it's likely urban runoff
df['waterbody_urban_runoff'] = ((df['waterbody_river'] == 0) & 
                               (df['waterbody_creek'] == 0) & 
                               (df['waterbody_coastal'] == 0) & 
                               ((df['asset_drainage_system'] == 1) | (df['protects_roadway'] == 1))).astype(int)

# --- 5. Technical Metadata ---
df['is_multi_phase'] = df['full_desc'].str.contains('phase|segment|section|package', na=False).astype(int)

# --- 6. Engineered Cross-Features ---
# Budget Tier Binning
bins = [0, 5_000_000, 20_000_000, 100_000_000, float('inf')]
labels = ['Micro (<5M)', 'Small (5M-20M)', 'Medium (20M-100M)', 'Large (>100M)']
df['budget_tier'] = pd.cut(df['ContractCost'], bins=bins, labels=labels)


/tmp/ipykernel_38940/1104292082.py:30: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['waterbody_coastal'] = df['full_desc'].str.contains(r'\b(sea|bay|gulf|coast|shore)\b', na=False).astype(int)


In [4]:
# Completion Season (Dry vs. Wet Season)
df['CompletionDateActual'] = pd.to_datetime(df['CompletionDateActual'], errors='coerce')
df['year_month'] = df['CompletionDateActual'].dt.to_period('M')

# 2. Add a counter for total projects
df['total_projects'] = 1

# 3. Create dummy variables for the budget_tier
df = pd.get_dummies(df, columns=["budget_tier"], prefix="budget_", dummy_na=True)

In [5]:
# validation: no columns should be tagged as nan
df.budget__nan.sum()

np.int64(0)

In [6]:
# 4. Drop the string categoricals that can't be summed (or you can dummy them first)
cols_to_drop = [
    'full_desc',
    'ContractCost',
    'CompletionDateActual', 
    'budget_tier', 
    'completion_season', 
    'ProjectDescription', 
    'ProjectComponentDescription',
    "budget__nan",
]
df_numeric = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# 5. Roll up to PSGC - Year - Month using SUM
# By default, grouping by psgc and year_month, then summing everything else.
rollup_df = df_numeric.groupby(['adm3_psgc', 'year_month']).sum().reset_index()

In [7]:
rollup_df.head()

,adm3_psgc,year_month,action_construction,action_rehabilitation,action_repair,action_improvement,action_extension,asset_drainage_system,asset_slope_protection,asset_bank_protection,...,waterbody_river,waterbody_creek,waterbody_coastal,waterbody_urban_runoff,is_multi_phase,total_projects,budget__Micro (<5M),budget__Small (5M-20M),budget__Medium (20M-100M),budget__Large (>100M)
0,102805000,2022-07,2,0,0,0,0,0,0,2,...,0,2,0,0,2,2,0,0,2,0
1,102805000,2022-12,1,0,0,0,0,0,0,0,...,1,1,0,0,0,1,0,0,1,0
2,102805000,2023-01,2,0,0,0,0,0,0,0,...,2,0,0,0,2,2,0,0,2,0
3,102805000,2023-03,4,0,0,0,0,0,0,0,...,0,4,0,0,2,4,0,0,4,0
4,102805000,2023-07,1,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0


### Creating the quarterly rollup

In [8]:
# Create an sequence of months from 2022-01 to 2025-12
full_months = pd.period_range(start="2022-01", end="2025-12", freq="M")

# Establish timeline index (0 to 47)
timeline_rank = list(range(len(full_months)))

# creating monthly_df
monthly_df = pd.merge(
    rollup_df[["adm3_psgc"]].drop_duplicates(), 
    pd.DataFrame({"year_month": full_months, "timeline_rank": timeline_rank}),
    how="cross"
)

display(monthly_df.shape)
monthly_df.head()

(6816, 3)

,adm3_psgc,year_month,timeline_rank
0,102805000,2022-01,0
1,102805000,2022-02,1
2,102805000,2022-03,2
3,102805000,2022-04,3
4,102805000,2022-05,4


In [9]:
monthly_merge_keys = ["adm3_psgc", "year_month"]
monthly_rollup_df = pd.merge(
    monthly_df,
    rollup_df,
    how="left",
    on=monthly_merge_keys
)
monthly_rollup_df = monthly_rollup_df.fillna(0)
display(monthly_rollup_df.shape)
monthly_rollup_df.head()

(6816, 29)

,adm3_psgc,year_month,timeline_rank,action_construction,action_rehabilitation,action_repair,action_improvement,action_extension,asset_drainage_system,asset_slope_protection,...,waterbody_river,waterbody_creek,waterbody_coastal,waterbody_urban_runoff,is_multi_phase,total_projects,budget__Micro (<5M),budget__Small (5M-20M),budget__Medium (20M-100M),budget__Large (>100M)
0,102805000,2022-01,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,102805000,2022-02,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,102805000,2022-03,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,102805000,2022-04,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,102805000,2022-05,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# Imputing and sorting by city and timeline rank before sliding windows
feature_cols = [col for col in rollup_df.columns if col not in ["adm3_psgc", "year_month"]]
monthly_rollup_df[feature_cols] = monthly_rollup_df[feature_cols].fillna(0)
monthly_rollup_df = monthly_rollup_df.sort_values(["adm3_psgc", "timeline_rank"]).reset_index(drop=True)

# Isolate feature tracking lists
lagged_dfs = []

# Slide windows per city
for city_id, group in monthly_rollup_df.groupby("adm3_psgc"):
    group_processed = group.copy()
    
    # Compute quarterly rollups
    for col in feature_cols:
        group_processed[f"{col}_roll_3M"] = group_processed[col].rolling(window=3, min_periods=1).sum()
        group_processed[f"{col}_roll_6M"] = group_processed[col].rolling(window=6, min_periods=1).sum()
        group_processed[f"{col}_roll_9M"] = group_processed[col].rolling(window=9, min_periods=1).sum()
        group_processed[f"{col}_roll_1Y"] = group_processed[col].rolling(window=12, min_periods=1).sum()
        
    # --- CRITICAL FIX: Explicitly copy the dataframe block here to defragment memory layout ---
    group_processed = group_processed.copy().drop(columns=["timeline_rank"])
        
    lagged_dfs.append(group_processed)

# Reassemble the dataframe
final_monthly_fcp_panel = pd.concat(lagged_dfs, ignore_index=True)
final_monthly_fcp_panel[final_monthly_fcp_panel["adm3_psgc"] == 102805000]

,adm3_psgc,year_month,action_construction,action_rehabilitation,action_repair,action_improvement,action_extension,asset_drainage_system,asset_slope_protection,asset_bank_protection,...,budget__Small (5M-20M)_roll_9M,budget__Small (5M-20M)_roll_1Y,budget__Medium (20M-100M)_roll_3M,budget__Medium (20M-100M)_roll_6M,budget__Medium (20M-100M)_roll_9M,budget__Medium (20M-100M)_roll_1Y,budget__Large (>100M)_roll_3M,budget__Large (>100M)_roll_6M,budget__Large (>100M)_roll_9M,budget__Large (>100M)_roll_1Y
0,102805000,2022-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,102805000,2022-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,102805000,2022-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,102805000,2022-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,102805000,2022-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,102805000,2022-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,102805000,2022-07,2.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,2.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0
7,102805000,2022-08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0
8,102805000,2022-09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,2.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0
9,102805000,2022-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0


In [15]:
fcp_folder = "flood_control_projects"
final_monthly_fcp_panel.to_csv(
    os.path.join(data_path, "urban_flood_control_projects.csv"),
    index=False
)